# GRPO and DPO: Qwen3.5-0.8B on Colab or Kaggle

Continues the SFT LoRA adapter with **GRPO** (`configs/grpo.yaml`), **DPO** (`configs/dpo.yaml`: sample pairs, then train) or **one GRPO ablation** (`configs/ablations/*.yaml`), and gives you the run directory as a zip.

- **GPU:** one 24 GB GPU (L4 / A10G) is what the configs are sized for. A 16 GB T4 runs with fp16 autocast, which is set automatically. If it runs out of memory, lower `max_completion_length`, then `per_device_train_batch_size` (ADR-002).
- **Inputs:** `data/processed/` (Claude references, RUNBOOK §1) and the SFT run's `checkpoints/sft/adapter/` (RUNBOOK §2).
- **Rewards:** four functions logged separately as `rewards/meaning`, `rewards/simplicity`, `rewards/entity_preservation` and `rewards/format`. The NLI reward model (`cross-encoder/nli-deberta-v3-base`) downloads on first use.
- **Nothing is pushed** to the Hugging Face Hub. Trackio logs locally; W&B is opt-in.

In [ ]:
!nvidia-smi

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/erfunm/Legal-Clause-Simplifier-using-RLHF.git"
BRANCH = "main"

STAGE = "grpo"  # "grpo", "dpo", or an ablation name such as "ablation-entity-off"

# Inputs: one zip holding data/processed/ and checkpoints/sft/adapter/, made locally with
#   zip -r rl_inputs.zip data/processed checkpoints/sft/adapter
#   "upload": upload that zip (Colab only); "kaggle": a Kaggle dataset holding the unzipped folders
INPUT_SOURCE = "upload"
KAGGLE_INPUT_DIR = "/kaggle/input/legal-clause-rl-inputs"

TRACKING = "trackio"  # "trackio" (local), "wandb" (needs the WANDB_API_KEY secret) or "none"

## 2. Install

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} repo
%cd repo
!pip install -q -e ".[train,tracking]"

## 3. Inputs

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

if INPUT_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()  # choose rl_inputs.zip
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(".")
elif INPUT_SOURCE == "kaggle":
    for sub in ("data/processed", "checkpoints/sft/adapter"):
        shutil.copytree(Path(KAGGLE_INPUT_DIR) / sub, sub, dirs_exist_ok=True)
else:
    raise ValueError(f"unknown INPUT_SOURCE {INPUT_SOURCE!r}")

for required in ("data/processed/train.jsonl", "checkpoints/sft/adapter/adapter_config.json"):
    assert Path(required).exists(), f"missing {required}"

## 4. Tracking (optional)

In [ ]:
if TRACKING == "wandb":
    try:
        from google.colab import userdata

        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except ImportError:
        from kaggle_secrets import UserSecretsClient

        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 5. Resolve the config and fit it to this GPU

In [ ]:
import yaml

from legal_simplifier.config import DPOConfigFile, GRPOConfigFile, load_config
from legal_simplifier.training.ablations import discover
from legal_simplifier.training.gpu import adapt_precision

if STAGE == "grpo":
    cfg = load_config("configs/grpo.yaml", GRPOConfigFile)
elif STAGE == "dpo":
    cfg = load_config("configs/dpo.yaml", DPOConfigFile)
else:
    (ablation,) = discover("configs/ablations", [STAGE])
    cfg = ablation.config
    print("Ablation changes:", {k: v for k, v in ablation.changes.items() if k != "experiment.name"})
cfg = adapt_precision(cfg)
cfg.tracking.backend = TRACKING
RUNTIME_CONFIG = f"configs/runtime_{cfg.stage}.yaml"
Path(RUNTIME_CONFIG).write_text(yaml.safe_dump(cfg.model_dump(mode="json"), sort_keys=False))
trainer_section = cfg.grpo if cfg.stage == "grpo" else cfg.dpo
RUN_DIR = trainer_section.output_dir
print(f"stage={cfg.stage} bf16={trainer_section.bf16} fp16={trainer_section.fp16} -> {RUN_DIR}")

## 6. Train

In [ ]:
if cfg.stage == "grpo":
    !lcs train-grpo --config {RUNTIME_CONFIG}
else:
    !lcs build-preferences --config {RUNTIME_CONFIG}
    !lcs train-dpo --config {RUNTIME_CONFIG}

## 7. Inspect the run: per-reward curves, peak memory, reward-hacking scan

In [ ]:
import json

history = [h for h in json.loads((Path(RUN_DIR) / "log_history.json").read_text()) if "loss" in h]
keys = [k for k in history[-1] if k.startswith("rewards/") and k.endswith("/mean")] + ["kl", "memory/max_allocated_gb"]
for row in history[:: max(1, len(history) // 10)] + history[-1:]:
    print(row.get("step"), {k: row[k] for k in keys if k in row})

In [ ]:
# DPO: scan the sampled outputs the pairs came from.
if cfg.stage == "dpo":
    !lcs detect-hacking --outputs {cfg.pairs.output_dir}/samples_train.jsonl

## 8. Download

Unzip into the repo on your machine. Later stages (evaluation, serving) load `<run>/merged/`.

In [ ]:
!zip -qr rl_run.zip {RUN_DIR} -x "*/checkpoint-*"
try:
    from google.colab import files

    files.download("rl_run.zip")
except ImportError:
    print(f"Saved {Path('rl_run.zip').resolve()}")